# BTCUSD 15m Transformer Training

Train the JackSparrow BTCUSD 15m **market-understanding** transformer in Google Colab (or locally with GPU).

Shared modules in `feature_store/transformer_btcusd_15m/` keep train/serve parity with `TransformerModelNode`. Training helpers live in `scripts/colab/transformer_training.py`.

**Pipeline:** Delta India OHLCV + funding/OI → causal features → forward-looking labels → purged train/val/test split → multi-task transformer → ONNX export.

**After training**, copy exports into the agent bundle:
- `btcusd_15m_transformer.onnx`
- `feature_config.json`
- → `agent/model_storage/JackSparrow_Transformer_BTCUSD/`

## Setup

In [ ]:
!pip install -q torch pandas numpy scikit-learn pyarrow onnx onnxruntime requests matplotlib

In [ ]:
# Clone the repo (uncomment on a fresh Colab runtime)
# !git clone https://github.com/energyforreal/JackSparrow.git
# %cd JackSparrow

import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "JackSparrow"]:
    if (candidate / "feature_store").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find feature_store/. Clone JackSparrow or open this notebook from the repo root."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Using repo root: {ROOT}")

## Configuration

In [ ]:
import json
from pathlib import Path

# --- Overrides (None = use DEFAULT_TRAINING_CONFIG) ---
export_dir = Path("/content/export")  # Path("export") for local runs
epochs = None
history_days = None
raw_cache_path = Path("btcusd_15m_raw.parquet")  # set None to skip parquet cache
refresh_data = False  # True forces API refetch even if cache exists

from feature_store.transformer_btcusd_15m.contract import (
    CONTINUOUS_LABEL_COLS,
    DEFAULT_TRAINING_CONFIG,
    FEATURE_COLS,
)
from feature_store.transformer_btcusd_15m.features import add_features
from feature_store.transformer_btcusd_15m.labels import compute_market_labels, trim_label_tail
from scripts.colab.transformer_data import fetch_history_bundle
from scripts.colab.transformer_training import set_training_seed

config = dict(DEFAULT_TRAINING_CONFIG)
if epochs is not None:
    config["epochs"] = epochs
if history_days is not None:
    config["history_days"] = history_days

set_training_seed(config["seed"])
print(json.dumps(config, indent=2))

## Data loading

In [ ]:
import pandas as pd

from feature_store.transformer_btcusd_15m.contract import HORIZON_RETURN_COLS, PATH_LABEL_COLS
from feature_store.transformer_btcusd_15m.labels import label_nan_summary
from scripts.colab.transformer_data import validate_derivatives_coverage

if raw_cache_path and raw_cache_path.is_file() and not refresh_data:
    print(f"Loading cached raw data from {raw_cache_path}")
    raw_df = pd.read_parquet(raw_cache_path)
    validate_derivatives_coverage(
        raw_df,
        min_coverage=config["min_derivatives_coverage"],
        warn_coverage=config["derivatives_coverage_warn"],
    )
else:
    raw_df = fetch_history_bundle(
        symbol=config["symbol"],
        resolution=config["resolution"],
        history_days=config["history_days"],
        base_url=config["base_url"],
        min_derivatives_coverage=config["min_derivatives_coverage"],
        derivatives_coverage_warn=config["derivatives_coverage_warn"],
    )
    if raw_cache_path:
        raw_df.to_parquet(raw_cache_path)
        print(f"Cached raw pull to {raw_cache_path}")

feat_df = add_features(raw_df, atr_period=config["atr_period"]).dropna().reset_index(drop=True)
feat_df = compute_market_labels(
    feat_df,
    return_horizon_bars=config["return_horizon_bars"],
    path_label_horizon_bars=config["path_label_horizon_bars"],
    mae_floor_atr_mult=config["mae_floor_atr_mult"],
)
feat_df = trim_label_tail(
    feat_df,
    return_horizon_bars=config["return_horizon_bars"],
    path_label_horizon_bars=config["path_label_horizon_bars"],
)

print(f"raw bars: {len(raw_df)}, feature rows: {len(feat_df)}")
nan_rates = label_nan_summary(feat_df)
print("return label NaN rate:", {k: nan_rates[k] for k in HORIZON_RETURN_COLS if k in nan_rates})
print("path label NaN rate:", {k: nan_rates[k] for k in PATH_LABEL_COLS if k in nan_rates})
assert len(feat_df) >= config["window_len"] + config["embargo_bars"] + 50, (
    "Too few rows after trim — increase history_days or check data pull"
)

## Windowing and purged split

Time-ordered train / val / test with embargo gaps (no random shuffle). Label stats and vol-regime quantiles are fit on **train only**.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

from scripts.colab.transformer_training import (
    build_windows,
    fit_label_stats,
    fit_vol_regime_edges,
    split_purged_windows,
    standardize_labels,
    to_vol_regime,
    WindowDataset,
)

x_all, y_all = build_windows(
    feat_df,
    FEATURE_COLS,
    CONTINUOUS_LABEL_COLS,
    config["window_len"],
    config["stride"],
)
splits = split_purged_windows(
    x_all,
    y_all,
    train_frac=config["train_frac"],
    val_frac=config["val_frac"],
    embargo_bars=config["embargo_bars"],
)

x_train, y_train = splits["x_train"], splits["y_train"]
x_val, y_val = splits["x_val"], splits["y_val"]
x_test, y_test = splits["x_test"], splits["y_test"]

label_mean, label_std = fit_label_stats(y_train)
y_train_z, m_train = standardize_labels(y_train, label_mean, label_std)
y_val_z, m_val = standardize_labels(y_val, label_mean, label_std)
y_test_z, m_test = standardize_labels(y_test, label_mean, label_std)

q_edges = fit_vol_regime_edges(y_train, config["vol_regime_quantiles"])
r_train = to_vol_regime(y_train, q_edges)
r_val = to_vol_regime(y_val, q_edges)
r_test = to_vol_regime(y_test, q_edges)

train_loader = DataLoader(
    WindowDataset(x_train, y_train_z, m_train, r_train),
    batch_size=config["batch_size"],
    shuffle=True,
    drop_last=True,
)
val_loader = DataLoader(
    WindowDataset(x_val, y_val_z, m_val, r_val),
    batch_size=config["batch_size"],
    shuffle=False,
)
test_loader = DataLoader(
    WindowDataset(x_test, y_test_z, m_test, r_test),
    batch_size=config["batch_size"],
    shuffle=False,
)

print(
    f"windows: train={len(x_train)} val={len(x_val)} test={len(x_test)} "
    f"(total={len(x_all)})"
)
print(f"vol regime edges: {q_edges}")
print(f"train regime balance: {np.bincount(r_train, minlength=4) / len(r_train)}")

## Model and training

In [ ]:
import torch

from scripts.colab.transformer_training import MarketTransformer, train_transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

model = MarketTransformer(
    n_features=len(FEATURE_COLS),
    d_model=config["d_model"],
    nhead=config["nhead"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    max_len=config["window_len"],
    n_continuous=len(CONTINUOUS_LABEL_COLS),
).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"model params: {param_count:,}")

result = train_transformer(
    model,
    train_loader,
    val_loader,
    config,
    device=device,
)
model.load_state_dict(result.model_state)
model.eval()

checkpoint_path = export_dir / "best_model.pt"
export_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {"model": result.model_state, "log_vars": result.log_vars, "config": config},
    checkpoint_path,
)
print(f"best val loss: {result.best_val_loss:.4f} (epoch {result.best_epoch})")
print(f"checkpoint: {checkpoint_path}")
print("learned log_vars:")
for name, lv in zip(CONTINUOUS_LABEL_COLS + ["vol_regime"], result.log_vars.numpy()):
    print(f"  {name:28s}  log_var={lv:+.3f}")

## Held-out test evaluation

Per-target MAE and correlation in real units (masked bars only). No P&L metrics — trading decisions are downstream in JackSparrow.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from scripts.colab.transformer_training import (
    evaluate_continuous_targets,
    evaluate_regime_head,
    print_return_horizon_metrics,
    print_target_metrics,
)

test_metrics = evaluate_continuous_targets(
    model,
    test_loader,
    device=device,
    label_mean=label_mean,
    label_std=label_std,
)
print_return_horizon_metrics(test_metrics)
print_target_metrics(test_metrics, title="Test set — all continuous targets:")

regime_pred, regime_true = evaluate_regime_head(model, test_loader, device=device)
print("\nVolatility regime confusion matrix:")
print(confusion_matrix(regime_true, regime_pred))
print(classification_report(regime_true, regime_pred, target_names=["LOW", "NORMAL", "HIGH", "EXTREME"]))

## Export ONNX and feature config

In [ ]:
from scripts.colab.transformer_training import export_transformer_bundle

onnx_path, cfg_path = export_transformer_bundle(
    model,
    export_dir,
    device=device,
    window_len=config["window_len"],
    n_features=len(FEATURE_COLS),
    feature_cols=FEATURE_COLS,
    label_mean=label_mean,
    label_std=label_std,
    q_edges=q_edges,
    config=config,
    verify=True,
)
print(f"Exported {onnx_path}")
print(f"Exported {cfg_path}")

## Download (Colab only)

Triggers browser downloads — skip this cell when running locally.

In [ ]:
try:
    from google.colab import files

    files.download(str(onnx_path))
    files.download(str(cfg_path))
    files.download(str(checkpoint_path))
except ImportError:
    print("Not in Colab — files are at:", export_dir)